In [1]:
# Ensure openpyxl is installed in the kernel environment
%pip install openpyxl --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
path = 'files/transactions.csv'

In [4]:
transactions = pd.read_csv(path)

In [5]:
type(transactions), transactions.shape

(pandas.DataFrame, (83488, 3))

In [6]:
transactions["date"] = pd.to_datetime(transactions["date"], errors="coerce")
 
transactions.dtypes

date            datetime64[us]
store_nbr                int64
transactions             int64
dtype: object

In [7]:
# Savaitės diena
transactions["weekday"] = transactions["date"].dt.day_name()
 
# Paprasta kategorija pagal transakcijų kiekį
transactions["activity_level"] = transactions["transactions"].apply(
    lambda x: "High" if x > 3000 else "Normal"
)
 
transactions.head()

,date,store_nbr,transactions,weekday,activity_level
0,2013-01-01,25,770,Tuesday,Normal
1,2013-01-02,1,2111,Wednesday,Normal
2,2013-01-02,2,2358,Wednesday,Normal
3,2013-01-02,3,3487,Wednesday,High
4,2013-01-02,4,1922,Wednesday,Normal


In [8]:
# Data konvertavimas į datetime
# Dažna klaida: palikti datas kaip 'object' ir vėliau gauti keistas filtravimo / grupavimo klaidas.
 
transactions["date"] = pd.to_datetime(transactions["date"], errors="coerce")
 
transactions.dtypes

date              datetime64[us]
store_nbr                  int64
transactions               int64
weekday                      str
activity_level               str
dtype: object

In [9]:
# Vidutinis transakcijų skaičius pagal parduotuvę (top 10)
avg_by_store = (
    transactions.groupby("store_nbr")["transactions"]
    .mean()
    .sort_values(ascending=False)
)
 
avg_by_store.head(10)

store_nbr
44    4336.966607
47    3897.322600
45    3697.742993
46    3571.921884
3     3201.879475
48    3045.787120
8     2767.285800
49    2727.550984
50    2614.456768
11    2370.219570
Name: transactions, dtype: float64

In [10]:
# Bendras transakcijų skaičius pagal datą (pirmos 10 datų)
daily_total = (
    transactions.groupby("date")["transactions"]
    .sum()
    .sort_values(ascending=False)
)
 
daily_total.head(10)

date
2015-12-24    171169
2016-12-24    167542
2016-12-23    156932
2014-12-24    156546
2013-12-24    155846
2015-12-23    153338
2013-12-23    145876
2014-12-23    144513
2015-12-22    138921
2016-12-22    138892
Name: transactions, dtype: int64

In [11]:
# Top 10 įrašų pagal didžiausią transakcijų skaičių
transactions.sort_values("transactions", ascending=False).head(10)

,date,store_nbr,transactions,weekday,activity_level
52011,2015-12-23,44,8359,Wednesday,High
71010,2016-12-23,44,8307,Friday,High
16570,2013-12-23,44,8256,Monday,High
33700,2014-12-23,44,8120,Tuesday,High
16572,2013-12-23,46,8001,Monday,High
16619,2013-12-24,46,7840,Tuesday,High
16573,2013-12-23,47,7727,Monday,High
52064,2015-12-24,44,7700,Thursday,High
33748,2014-12-24,44,7689,Wednesday,High
70904,2016-12-21,44,7597,Wednesday,High


In [12]:
# Santrauka pagal parduotuvę
summary_by_store = (
    transactions.groupby("store_nbr")
    .agg(
        days=("date", "nunique"),
        total_transactions=("transactions", "sum"),
        avg_transactions=("transactions", "mean"),
        max_transactions=("transactions", "max"),
    )
    .sort_values("total_transactions", ascending=False)
    .reset_index()
)
 
summary_by_store.head(10)

,store_nbr,days,total_transactions,avg_transactions,max_transactions
0,44,1677,7273093,4336.966607,8359
1,47,1677,6535810,3897.322600,7727
2,45,1677,6201115,3697.742993,7305
3,46,1677,5990113,3571.921884,8001
4,3,1676,5366350,3201.879475,6085
5,48,1677,5107785,3045.787120,7044
6,8,1676,4637971,2767.285800,5261
7,49,1677,4574103,2727.550984,6600
8,50,1677,4384444,2614.456768,5456
9,11,1676,3972488,2370.219570,5018


In [13]:
# Eksportas į Excel su keliais lapais
output_file = "transactions_analysis.xlsx"
 
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    transactions.to_excel(writer, sheet_name="data", index=False)
    summary_by_store.to_excel(writer, sheet_name="summary", index=False)
 
output_file

'transactions_analysis.xlsx'